# Random LTL formula generation

Generates `F` random formulas for every `(N, L, P)` combination (`N`
propositions, `L` symbols, `P` probability of picking a temporal operator) and
writes them to `FORMULAS_FILE`, one per line as
`N<TAB>L<TAB>P<TAB>index<TAB>formula`. Run the experiments on them with
`Random_formula_experiment.ipynb`.

In [1]:
import random

alphabet = ['a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z']
unary = ['!', 'X ']
binary_non_temporal = ['&&', '||']
binary_temporal = ['U','W']

In [2]:
def op_weights(P):
    # unary and non-temporal binary operators share 1-P, temporal ones share P
    return [(1-P)/4]*4 + [P/2]*2

In [3]:
# Sweep parameters
N_VALUES = [1, 2, 3]
L_VALUES = [i*5 for i in range(1,10)]
P_VALUES = [1/3, 1/2, .7, .95]
F = 100  # number of random formulas per (N, L, P) combination

# None = a different set on every run; an int makes the set reproducible
SEED = None

FORMULAS_FILE = "formulas.txt"

# generate_formula() reads N and weights from the global scope; they are reset
# inside the sweep
N = N_VALUES[0]
weights = op_weights(P_VALUES[0])

In [4]:
def generate_formula(L):
    if L == 1:
        return random.choice(alphabet[:N])

    op = random.choices(unary+binary_non_temporal+binary_temporal, weights)[0]
    if L == 2:
        op = random.choice(unary)

    if op in unary:
        return '('+op+generate_formula(L-1)+')'

    S = random.randrange(1,L-1)

    return '(' + generate_formula(S) + ' ' + op + ' ' + generate_formula(L-S-1) + ')'

In [5]:
random.seed(SEED)

formulas = []  # (N, L, P, index, formula)
for n in N_VALUES:
    N = n  # used by generate_formula
    for L in L_VALUES:
        for p in P_VALUES:
            weights = op_weights(p)  # used by generate_formula
            formulas.extend((n, L, p, i, generate_formula(L)) for i in range(F))

with open(FORMULAS_FILE, "w") as fh:
    for n, L, p, i, formula in formulas:
        fh.write(f"{n}\t{L}\t{p}\t{i}\t{formula}\n")

print(f"Wrote {len(formulas)} formulas to {FORMULAS_FILE}")

Wrote 10800 formulas to formulas.txt
